# Quick Start Guide

Generate forecasting datasets from news articles in minutes.

Lightning Rod automatically creates training data by:
1. Collecting news articles from a time period
2. Generating forecasting questions from those articles
3. Finding the answers automatically using web search
4. Returning a dataset ready for model training

This uses the "future as label" approach: we generate questions about future events, then use what actually happened as the ground truth labels.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Build a pipeline

A pipeline has three components:

1. **Seed Generator** - Collects news articles from a time period
2. **Question Generator** - Creates forecasting questions from the articles
3. **Labeler** - Finds the answers automatically using web search

Let's build a simple pipeline:

In [3]:
from datetime import datetime
from lightningrod import (
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    WebSearchLabeler,
    QuestionPipeline,
    BinaryAnswerType,
)

seed_generator = NewsSeedGenerator(
    start_date=datetime(2025, 10, 1),
    end_date=datetime(2025, 11, 1),
    search_query="technology announcements",
)

answer_type = BinaryAnswerType()

question_generator = ForwardLookingQuestionGenerator(
    instructions="Generate forward-looking questions about technology announcements.",
    answer_type=answer_type,
)

# Labeler automatically finds answers to questions using web search
labeler = WebSearchLabeler(answer_type=answer_type)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    labeler=labeler,
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. The `max_questions` parameter limits how many questions to generate (useful for testing).

In [4]:
dataset = lr.transforms.run(pipeline, max_questions=20, name="Quick start")

Output()

> Note: This can take a few minutes to complete processing.

## View the results

Each sample in the dataset contains:
- The original news article
- A forecasting question generated from it
- The answer (found via web search) with confidence score
- A formatted prompt ready for model training

View results as a data frame:

In [5]:
%pip install pandas

from IPython.display import clear_output
clear_output()

import pandas as pd

samples = dataset.download()
rows = dataset.flattened()
df = pd.DataFrame(rows)

print(f"Generated {dataset.num_rows} samples (%.1f%% valid)\n" % (dataset.valid_count() / dataset.num_rows * 100))

cols = ["question_text", "answer", "label_confidence", "is_valid", "invalid_reason"]
df[[c for c in cols if c in df.columns]]

Generated 20 samples (55.0% valid)



,question_text,label_confidence
0,Will Nvidia CEO Jensen Huang officially announ...,NaN
1,Will Apple Inc. officially release a MacBook P...,1.00
2,Will the 'Manifesto of the First China (Hangzh...,0.95
3,Will ESPN or Disney-owned platforms broadcast ...,1.00
4,Will at least one of the Nike Mind footwear mo...,1.00
5,Will General Motors release a Cadillac Escalad...,1.00
6,"By December 31, 2026, will AMD officially laun...",0.95
7,Will Diligent Robotics have its Moxi robots de...,1.00
8,Will the India-AI Impact Summit 2026 take plac...,1.00
9,Will the Chinese government's export restricti...,1.00


## Next steps

- **Different data sources**: See examples 02/03 for news and custom document sources
- **Different question types**: See examples 04-07 for different question/answer types
- **Full API reference**: See [API.md](../API.md) for all options and configurations